
1. What is data ingestion?
Interview definition:

Data ingestion is the process of collecting data from different source systems and loading it into a centralized storage or data processing platform such as Databricks.

Real-time example:

An e-commerce company receives data from:
- MySQL – Customers and Products.
- Oracle – Orders and Payments.
- CSV files – Shipments and Inventory.
- APIs – Customer activity and payment status.
- Kafka – Real-time order events.
These datasets are ingested into Databricks for further processing.








medium.com


2. What is Medallion Architecture?
Interview definition:

Medallion Architecture is a data design pattern that organizes data into three layers: Bronze, Silver and Gold.

Each layer progressively improves data quality, structure and business usability.








figlab.ai


2.1 Bronze Layer
Definition: The Bronze layer stores raw ingested data from source systems with minimal transformation.

Characteristics:
- Raw data storage.
- Append-oriented and generally immutable.
- Schema-on-read where appropriate.
- Maintains source information.
- Supports auditing and reprocessing.
Example:
df = spark.read.csv(    "/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/customers/customers.csv",    header=True,    inferSchema=True)df.write.format("delta") \    .mode("append") \    .saveAsTable("workspace.ecommerce_bronze.customers")



Interview point: Bronze is not necessarily physically immutable. We design it to preserve original ingested records and avoid destructive transformations.
2.2 Silver Layer
Definition: The Silver layer contains cleaned, validated, deduplicated and standardized data.

Characteristics:
- Removes duplicate records.
- Handles missing values.
- Converts columns into correct data types.
- Applies data quality rules.
- Standardizes formats.
- Performs required joins and referential checks.
Example:
from pyspark.sql.functions import coldf = spark.table("workspace.ecommerce_bronze.customers")silver_df = (    df.dropDuplicates(["CustomerID"])      .filter(col("CustomerID").isNotNull())      .withColumn("CustomerID", col("CustomerID").cast("int")))silver_df.write.format("delta") \    .mode("overwrite") \    .option("overwriteSchema", "true") \    .saveAsTable("workspace.ecommerce_silver.customers")



Interview point: Silver data should be trusted and reusable across different business requirements.
2.3 Gold Layer
Definition: The Gold layer contains business-ready, aggregated and analytical datasets used for reporting, dashboards and decision-making.

Characteristics:
- Business-level aggregations.
- KPI calculations.
- Dimensional models and data marts.
- Dashboard-ready datasets.
- Optimized for analytical queries.
Example:
from pyspark.sql.functions import sum, countdf = spark.table("workspace.ecommerce_silver.orders")gold_df = (    df.groupBy("OrderDate")      .agg(          count("OrderID").alias("TotalOrders"),          sum("TotalAmount").alias("TotalRevenue")      ))gold_df.write.format("delta") \    .mode("overwrite") \    .option("overwriteSchema", "true") \    .saveAsTable("workspace.ecommerce_gold.daily_sales")



Interview point: Gold is where business users consume trusted data through Databricks SQL dashboards, Power BI or other reporting tools.
3. Bronze vs Silver vs Gold
Feature	Bronze	Silver	Gold
Data	Raw	Cleaned	Business-ready
Quality	Source quality	Validated	Trusted
Duplicates	Usually retained	Removed/handled	Aggregated
Schema	Source-oriented	Standardized	Business-oriented
Operations	Append-oriented	Merge/update	Aggregation
Users	Data engineers	Data engineers	Analysts/business
Purpose	Historical preservation	Data quality	Analytics

4. What is Auto Loader?
Interview definition:

Auto Loader is a Databricks feature that incrementally and efficiently ingests new files from cloud storage into Spark DataFrames or Delta tables using the cloudFiles format.

It tracks discovered files using checkpoint information.

Example:

An e-commerce company receives 100 new order CSV files every day. Instead of manually identifying and loading each file, Auto Loader detects and processes new files incrementally.
source_path = "/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/orders"df = (    spark.readStream    .format("cloudFiles")    .option("cloudFiles.format", "csv")    .option("header", "true")    .option("cloudFiles.schemaLocation",            "/Volumes/workspace/default/data_volume/ecommerce_lakehouse/schemas/orders")    .load(source_path))



4.1 Auto Loader Directory Listing Mode
Definition: Auto Loader discovers files by listing the source directory and identifying new files.

Example:
- Source directory contains 100 files.
- Auto Loader discovers files.
- It processes files not already recorded in its checkpoint.
- Newly arriving files are picked up in subsequent processing.
4.2 Auto Loader File Notification Mode
Definition: Auto Loader uses cloud storage notifications to discover new files instead of relying entirely on directory listing.

Examples of cloud notification infrastructure:
- AWS S3 – SQS notifications.
- Azure Data Lake Storage – Event Grid.
- Google Cloud Storage – Pub/Sub notifications.
Directory Listing	File Notification
Discovers files by listing directories	Discovers files through notifications
No notification infrastructure required	Requires notification infrastructure
Suitable for simpler setups	Suitable for large-scale ingestion
Can involve repeated listing	Reduces discovery overhead

Important: Auto Loader's exact discovery behavior and configuration depend on the cloud platform and Databricks runtime.
5. What is COPY INTO?
Interview definition:

COPY INTO is a Databricks SQL command used to incrementally load files from cloud storage into an existing Delta table.

It keeps track of previously loaded files and avoids loading the same files again under normal operation.

Example:
COPY INTO workspace.ecommerce_bronze.orders
FROM '/Volumes/workspace/default/data_volume/ecommerce_lakehouse/landing/orders'
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'true'
);


5.1 Auto Loader vs COPY INTO
Feature	Auto Loader	COPY INTO
Processing	Incremental	Incremental
Interface	PySpark Structured Streaming	SQL
New files	Automatically discovered	Discovered when command runs
Checkpoint	Streaming checkpoint	Load history
Schema evolution	Supported	More limited
Best use	Continuous ingestion	Scheduled/batch ingestion

Interview question: When should you use Auto Loader instead of COPY INTO?

Answer:

I use Auto Loader when I need scalable, continuous or frequent incremental file ingestion, schema evolution and checkpoint-based processing. I use COPY INTO when I need a simpler SQL-based incremental batch ingestion process.
6. What is incremental loading?
Definition:

Incremental loading is the process of loading only new or changed records instead of reloading the complete dataset every time.

Real-time example:

Suppose an e-commerce company has 1 million order records.
Day	New records
Day 1	100,000
Day 2	10,000
Day 3	15,000
Day 4	8,000

Instead of processing all 125,000 records again on Day 4, we process only the new 8,000 records.

Example:
source_df = spark.readStream \    .format("cloudFiles") \    .option("cloudFiles.format", "csv") \    .option("header", "true") \    .load(source_path)



Auto Loader discovers new files. The streaming checkpoint maintains file-processing progress.

Important interview distinction:

Incremental file ingestion does not automatically mean incremental row-level updates. If an existing file contains updated records, you need an appropriate CDC or MERGE strategy to apply those changes.
7. What is idempotency?
Frequently Asked
Definition:

Idempotency means executing the same data pipeline multiple times produces the same final result without creating duplicate records or incorrect data.

Real-time example:

Suppose an order file contains:
OrderID	Amount
101	500
102	700

The pipeline processes this file.

Due to a job failure, the same batch is retried.

Without idempotency, the records might be inserted twice.

With idempotency, the final table still contains only one record for each order.
7.1 How do you implement idempotency?
Common methods:
1. Auto Loader checkpoints.
2. Deduplication using business keys.
3. Delta Lake MERGE.
4. Batch IDs or unique transaction IDs.
5. Controlled checkpoint and retry management.
Example using MERGE:
MERGE INTO workspace.ecommerce_silver.orders AS target

USING workspace.ecommerce_bronze.orders AS source

ON target.OrderID = source.OrderID

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;


Important: This assumes one source record per OrderID. If the source has multiple records for the same key, deduplicate it first and define which record should win.

Also, an upsert is not automatically idempotent if your transformations use non-deterministic values or repeatedly increment existing values.

8. Traditional Data Warehouse vs Medallion Architecture
Traditional Data Warehouse	Medallion Architecture
Usually ETL	Usually ELT
Transform before loading final warehouse	Load raw data first, then transform
Staging area	Bronze
ODS / integrated warehouse	Primarily Silver
Data marts / reporting	Gold
Relational tables	Delta tables
Structured source data	Structured, semi-structured and other supported data

8.1 What is ETL?
ETL stands for:
- Extract – Collect data from source systems.
- Transform – Clean and transform data.
- Load – Load transformed data into the target warehouse.
8.2 What is ELT?
ELT stands for:
- Extract – Collect source data.
- Load – Load raw data into the lakehouse.
- Transform – Transform data inside Databricks using Spark SQL or PySpark.
Interview answer:

In traditional ETL, transformation happens before loading into the target data warehouse. In a modern Databricks lakehouse, we commonly use ELT, where raw data is first loaded into Bronze, cleaned in Silver and aggregated in Gold.
9. DWH to Medallion Architecture Mapping
This is an important interview comparison.
Traditional DWH	Databricks equivalent
DSA (Data Staging Area)	Bronze
ODS (Operational Data Store)	Silver
Integrated Data Warehouse	Silver
Data Mart	Gold
Reports	Gold / BI dashboards
MySQL / Oracle tables	Delta tables
Linux file system	Cloud object storage / Volumes
Python scripts	PySpark
SQL transformations	Spark SQL
Traditional batch jobs	Databricks Workflows

Note: This is a conceptual mapping, not a strict one-to-one migration rule. The actual design depends on business requirements.

10. Design an Auto Loader to Medallion Pipeline
Very High Priority – Scenario Based
Interview question:

How would you design an end-to-end Databricks pipeline using Auto Loader and Medallion Architecture?
10.1 Business requirement
An e-commerce company receives seven datasets:
- Customers
- Products
- Orders
- Order Items
- Payments
- Shipments
- Inventory Snapshots
The requirement is to ingest new CSV files automatically, validate data, remove duplicates, apply business rules and generate analytical datasets for dashboards.
10.2 End-to-end architecture








linkedin.com


SOURCE SYSTEMS

MySQL · Oracle · CSV · APIs


LANDING LAYER

Incoming CSV / JSON / Parquet files

Auto Loader / COPY INTO

BRONZE
Raw Delta tables · Source metadata · Append-oriented


SILVER
Deduplication · Data types · Validation · Standardization · MERGE


GOLD
Daily Sales · Product Performance · Customer Lifetime Value · KPIs


CONSUMPTION

Databricks SQL · Power BI · Business Dashboards


10.3 Pipeline execution steps
Step	Operation	Purpose
1	Receive source files	Collect data
2	Auto Loader	Detect new files
3	Bronze ingestion	Preserve raw records
4	Silver transformation	Clean and validate
5	Deduplication	Remove repeated business records
6	MERGE	Apply inserts and updates
7	Gold aggregation	Calculate KPIs
8	SQL Dashboard	Business reporting

10.4 Recommended project folder structure
Instead of putting the entire pipeline in one notebook, create a modular notebook structure.
E-Commerce-Lakehouse/
│
├── 00_configuration/
│   └── config
│
├── 01_libraries/
│   └── common_functions
│
├── 02_bronze/
│   └── bronze_main
│
├── 03_silver/
│   └── silver_main
│
├── 04_gold/
│   └── gold_main
│
├── 05_orchestration/
│   └── main_pipeline
│
└── 06_monitoring/
    └── pipeline_monitoring


10.5 Configuration notebook
Purpose: Maintain all reusable paths, table names and dataset configurations in one place.

Example:
CATALOG = "workspace"BRONZE_SCHEMA = "ecommerce_bronze"SILVER_SCHEMA = "ecommerce_silver"GOLD_SCHEMA = "ecommerce_gold"VOLUME_PATH = (    "/Volumes/workspace/default/data_volume/"    "ecommerce_lakehouse")LANDING_PATH = f"{VOLUME_PATH}/landing"CHECKPOINT_PATH = f"{VOLUME_PATH}/checkpoints"SCHEMA_PATH = f"{VOLUME_PATH}/schemas"DATASETS = [    "customers",    "products",    "orders",    "order_items",    "payments",    "shipments",    "inventory_snapshots"]



10.6 Common functions notebook
Purpose: Store reusable functions to avoid repeating code.

Example:
def get_table_name(layer, dataset):    return f"workspace.ecommerce_{layer}.{dataset}"def write_delta(df, table_name, mode="append"):    (        df.write        .format("delta")        .mode(mode)        .saveAsTable(table_name)    )



Other functions you can add:
- read_autoloader()
- write_bronze()
- clean_silver()
- remove_duplicates()
- validate_records()
- merge_delta()
- create_gold()
10.7 Bronze main notebook
Purpose: Read incoming files and store raw data.
dbutils.widgets.text("dataset", "orders")dataset = dbutils.widgets.get("dataset")source_path = f"{LANDING_PATH}/{dataset}"checkpoint_path = f"{CHECKPOINT_PATH}/{dataset}"schema_path = f"{SCHEMA_PATH}/{dataset}"bronze_table = get_table_name("bronze", dataset)df = (    spark.readStream    .format("cloudFiles")    .option("cloudFiles.format", "csv")    .option("header", "true")    .option("cloudFiles.schemaLocation", schema_path)    .load(source_path))(    df.writeStream    .format("delta")    .option("checkpointLocation", checkpoint_path)    .option("mergeSchema", "true")    .trigger(availableNow=True)    .toTable(bronze_table)



Here, availableNow=True processes the available new files and then stops. A Databricks Job can execute it again on the next scheduled run.
10.8 Silver main notebook
Purpose: Clean, validate, deduplicate and standardize Bronze data.
from pyspark.sql.functions import coldbutils.widgets.text("dataset", "orders")dataset = dbutils.widgets.get("dataset")bronze_table = get_table_name("bronze", dataset)silver_table = get_table_name("silver", dataset)df = spark.table(bronze_table)clean_df = (    df.dropDuplicates()      .filter(col("OrderID").isNotNull()))(    clean_df.write    .format("delta")    .mode("overwrite")    .saveAsTable(silver_table))



This is a simplified batch example. In a production incremental pipeline, use a stable business key, a deterministic record-selection rule and Delta MERGE rather than overwriting the entire Silver table on every run.
10.9 Gold main notebook
Purpose: Generate business KPIs and analytical tables.
from pyspark.sql.functions import sum, countsilver_table = "workspace.ecommerce_silver.orders"gold_table = "workspace.ecommerce_gold.daily_sales"df = spark.table(silver_table)gold_df = (    df.groupBy("OrderDate")      .agg(          count("OrderID").alias("TotalOrders"),          sum("TotalAmount").alias("TotalRevenue")      ))(    gold_df.write    .format("delta")    .mode("overwrite")    .option("overwriteSchema", "true")    .saveAsTable(gold_table))



This is a full-refresh example. For larger datasets, incremental Gold processing can use affected-date recomputation or MERGE.

11. %run vs dbutils.notebook.run()
This is a common Databricks interview question.
11.1 %run
%run executes another notebook and makes its variables and functions available in the current notebook's context.

Example:
%run "../00_configuration/config"



%run "../01_libraries/common_functions"



Now you can use:
print(BRONZE_SCHEMA)table_name = get_table_name("bronze", "orders")



11.2 dbutils.notebook.run()
It executes another notebook as a separate notebook run and allows parameters to be passed.

Example:
result = dbutils.notebook.run(    "../02_bronze/bronze_main",    0,    {"dataset": "orders"})print(result)



Inside the child notebook:
dbutils.widgets.text("dataset", "orders")dataset = dbutils.widgets.get("dataset")print(f"Processing dataset: {dataset}")dbutils.notebook.exit("SUCCESS")



11.3 Difference
Feature	%run	dbutils.notebook.run()
Execution	Same notebook context	Separate notebook run
Variables	Shared	Not directly shared
Parameters	Not directly passed	Supported
Return value	No explicit return	notebook.exit()
Best use	Import configurations/functions	Notebook orchestration
Reusability	Code reuse	Workflow execution

My recommendation: Use %run for shared configurations and functions. Use Databricks Workflows for production orchestration, with separate notebook tasks and task dependencies. dbutils.notebook.run() is useful for programmatic notebook execution, but it is not a replacement for Workflows.
12. How do you refactor a monolithic notebook?
Important Scenario Question
Interviewer question:

Suppose you have one notebook containing 1,000 lines of code for ingestion, cleaning and aggregation. How would you convert it into a reusable framework?

Answer:

First, I identify the responsibilities of the existing notebook and separate them into logical modules.

My approach:
1. Extract all paths, schemas and table names into a configuration notebook.
2. Move common transformations into a function library.
3. Create a separate Bronze notebook for ingestion.
4. Create a Silver notebook for data cleaning and validation.
5. Create a Gold notebook for aggregations.
6. Create an orchestration notebook or Databricks Workflow to execute the notebooks.
7. Pass dataset names and other parameters dynamically.
8. Add logging, checkpoints, error handling and monitoring.
Before refactoring
Single monolithic notebook
One notebook – 1,000 lines
- Configuration
- Auto Loader
- Bronze
- Silver
- Gold
- Logging
- Error handling

Problems: Difficult maintenance, duplicated code, difficult debugging and limited reusability.

After refactoring
Modular framework
Configuration + Common Functions

Bronze

Ingestion


Silver

Cleaning


Gold

Analytics




Workflow Orchestration

Parameters · Dependencies · Retries · Monitoring


Benefits:
- Easy maintenance.
- Reusable functions.
- Independent testing.
- Better debugging.
- Dynamic dataset processing.
- Easier scaling from 1 dataset to 100 datasets.
13. Generic plug-and-play ingestion framework
Interviewer question:

How would you build a generic framework to ingest 100 datasets without creating 100 separate notebooks?

Answer:

I would create a configuration-driven framework.

Instead of hardcoding every dataset, I maintain a configuration dictionary.

Example:
DATASET_CONFIG = {    "customers": {        "format": "csv",        "primary_key": "CustomerID",        "load_type": "incremental"    },    "orders": {        "format": "csv",        "primary_key": "OrderID",        "load_type": "incremental"    },    "products": {        "format": "csv",        "primary_key": "ProductID",        "load_type": "incremental"    }}



Now create a generic processing function:
def process_dataset(dataset):    config = DATASET_CONFIG[dataset]    print(f"Processing: {dataset}")    print(f"Format: {config['format']}")    print(f"Primary Key: {config['primary_key']}")    print(f"Load Type: {config['load_type']}")



Execute:
for dataset in DATASET_CONFIG:    process_dataset(dataset)



Output:
Processing: customers
Format: csv
Primary Key: CustomerID
Load Type: incremental

Processing: orders
Format: csv
Primary Key: OrderID
Load Type: incremental

Processing: products
Format: csv
Primary Key: ProductID
Load Type: incremental


This is only the configuration and dispatch skeleton. A production framework would also define source paths, target tables, schema expectations, incremental keys, transformation rules, quarantine policies and checkpoint locations.
14. Top 15 interview questions with answers
These are the questions you should practise verbally.

Q1. What is Medallion Architecture?

It is a multi-layer data architecture consisting of Bronze, Silver and Gold layers that progressively improve data quality and usability.

Q2. Why do we need three layers?

To separate raw ingestion, data quality processing and business-level analytics.

Q3. What is the difference between Bronze and Silver?

Bronze stores raw source data, whereas Silver stores cleaned, validated and standardized data.

Q4. Why is Bronze generally append-only?

To preserve historical source data, support auditing and allow reprocessing.

Q5. What is schema evolution?

Schema evolution allows a Delta table or ingestion process to accommodate supported schema changes, such as newly added columns.

Q6. What happens when a new column arrives in Auto Loader?

Depending on the configured schema evolution mode, Auto Loader can detect the new column and evolve its schema or stop with a schema-change error for handling.

Q7. What is the difference between batch and streaming ingestion?

Batch processes data at scheduled intervals, while streaming processes incoming data continuously or incrementally.

Q8. What is a checkpoint?

A checkpoint stores streaming progress and state so the stream can recover after failures.

Q9. Can Auto Loader process the same file twice?

Normally, it tracks discovered files using checkpoint information. Reprocessing can occur if checkpoint state is lost, reset or changed.

Q10. What is the difference between full load and incremental load?

Full load reloads the complete dataset; incremental load processes only new or changed data.

Q11. How do you handle duplicate records?

I use business keys, window functions, deterministic deduplication and Delta MERGE.

Q12. What is idempotency?

It ensures that retrying a pipeline does not create duplicate or incorrect results.

Q13. What is the purpose of Gold?

Gold provides business-ready datasets for KPIs, reporting and dashboards.

Q14. Why do you use a configuration-driven framework?

It reduces hardcoding, improves reusability and allows the same framework to process multiple datasets dynamically.

Q15. How do you orchestrate a Medallion pipeline?

I use Databricks Workflows with Bronze, Silver and Gold tasks connected through dependencies, parameters, retries and monitoring.